# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook demonstrates step-by-step exploration and processing of the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json) using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library. All dataset elements (record sets, fields, columns) are referenced by their `@id` identifiers for clarity and reproducibility.

### Dataset Source
The dataset is described by a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load dataset metadata and records from the Croissant-defined source using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset via mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access metadata object
metadata = dataset.metadata

# Print dataset title and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Let's inspect all available record sets, fields, and columns using their `@id` values.

Below, we enumerate all record sets and their fields by `@id`.

In [ ]:
from pprint import pprint

# List all RecordSets available by @id and name
record_sets = dataset.record_sets
if not record_sets:
    print("No record sets defined directly in metadata; attempting to infer from records...")

else:
    print("Available record sets:")
    for rs in record_sets:
        print(f"- @id: {rs.id}, name: {getattr(rs, 'name', 'N/A')}")
        if hasattr(rs, 'fields'):
            print("  Fields:")
            for f in rs.fields:
                print(f"    - @id: {f.id}, name: {getattr(f, 'name', 'N/A')}")
        elif hasattr(rs, 'columns'):
            print("  Columns:")
            for col in rs.columns:
                print(f"    - @id: {col.id}, name: {getattr(col, 'name', 'N/A')}")

To further explore, let's enumerate all available record set `@id` values in this dataset. We'll print records for a discovered record set as an example.

In [ ]:
# Attempt to enumerate all record_set identifiers
record_set_ids = []
if hasattr(dataset, "record_sets") and dataset.record_sets:
    for rs in dataset.record_sets:
        record_set_ids.append(rs.id)
else:
    # Fallback: try to infer from manifest or records
    # This uses a protected member - if mlcroissant is updated, this may change.
    try:
        record_sets_list = list(dataset.list_record_sets())
        for rec in record_sets_list:
            record_set_ids.append(rec)
    except Exception as e:
        print("Could not enumerate record sets.")

if len(record_set_ids) == 0:
    print("No record sets found.")
else:
    print("Found record_set @ids:")
    pprint(record_set_ids)
    # Display a few sample records from the first record set
    preview_id = record_set_ids[0]
    print(f"\nPreview of records from record_set '{preview_id}':")
    for i, rec in enumerate(dataset.records(record_set=preview_id)):
        pprint(rec)
        if i >= 2:
            break

## 3. Data Extraction

We will extract all records from each record set into a pandas DataFrame. All DataFrames will be referenced by their record set `@id`.

_Note: Replace `<record_set_id>` in the code below with the discovered `@id` from the previous step._

In [ ]:
# Extract all available record sets into DataFrames by their @id
dataframes = {}

for rid in record_set_ids:
    records = list(dataset.records(record_set=rid))
    dataframes[rid] = pd.DataFrame(records)
    print(f"Loaded record_set '{rid}' with shape {dataframes[rid].shape}")

# Example: list columns for the first record set
main_record_set_id = record_set_ids[0]
print(f"Columns in record_set '{main_record_set_id}':")
print(dataframes[main_record_set_id].columns.tolist())

# Display first 5 records
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Here we demonstrate standard data processing steps, including filtering, normalization, and grouping, using field `@id` values for all references.

_Replace the `numeric_field_id` and `group_field_id` below with those relevant from the record set column listing above._

In [ ]:
# Example: suppose we found a numeric field '@id' called 'coefficient' for regression coefficients
# and a groupable field '@id' called 'variable'.

# Replace the below with actual @id strings found in the previous listing, e.g.:
# numeric_field_id = 'https://api.app.sen.science/frontiers/7853015/fields/coefficient'
# group_field_id = 'https://api.app.sen.science/frontiers/7853015/fields/variable'

numeric_field_id = None
group_field_id = None

for col in dataframes[main_record_set_id].columns:
    # Try to guess which columns are numeric and which could be grouping variables
    if numeric_field_id is None and ('coef' in col or 'loglikelihood' in col or 'std' in col):
        numeric_field_id = col
    if group_field_id is None and ('var' in col or 'group' in col or 'field' in col):
        group_field_id = col

if numeric_field_id is None:
    print('No likely numeric field was found in columns. Please define "numeric_field_id" explicitly.')
else:
    print(f"Using numeric field '@id': {numeric_field_id}")
    
    df = dataframes[main_record_set_id]

    # Filtering, for demonstration: exclude nulls and keep values greater than a threshold
    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize the selected numeric field
    mu, sigma = filtered_df[numeric_field_id].mean(), filtered_df[numeric_field_id].std()
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - mu) / sigma
    print(f"\nNormalized column '{numeric_field_id}':")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Grouping by the chosen group_field_id (if available)
    if group_field_id and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nMean {numeric_field_id} grouped by '{group_field_id}':")
        print(grouped_df.head())

## 5. Visualization

Visualize numeric distributions or key relationships from the data using field `@id`s. Adjust the chart as needed per available fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example histogram and boxplot for numeric field
if numeric_field_id is not None and numeric_field_id in dataframes[main_record_set_id].columns:
    plt.figure(figsize=(10,4))
    sns.histplot(dataframes[main_record_set_id][numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Histogram of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id and group_field_id in dataframes[main_record_set_id].columns:
        plt.figure(figsize=(12,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=dataframes[main_record_set_id].dropna(subset=[numeric_field_id, group_field_id]))
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

else:
    print("No numeric field found for plotting. Update the variable 'numeric_field_id' with the @id of a numeric column.")

## 6. Conclusion

We have successfully loaded, explored, and visualized the FAIR^2 dataset with `mlcroissant`, using field-level `@id` references. This approach ensures transparency, reproducibility, and clarity in data processing. For further analysis, refer to more detailed field and schema information by inspecting the dataset's Croissant manifest and metadata attributes.